In [1]:
import pandas as pd
import matplotlib.pyplot as plt
from joblib import Parallel, delayed
import numpy as np
import re
from bs4 import BeautifulSoup
from geopy.geocoders import ArcGIS
import requests


## 1. Download race records from https://www.beermile.com

In [2]:
def fetch_record(record_id):
    url = f"https://www.beermile.com/records/{record_id}?official=true&event=1"
    try:
        tables = pd.read_html(url)
        r = requests.get(url)
        soup = BeautifulSoup(r.text, "html.parser")

        race_links = ["https://www.beermile.com" + a["href"] for a in soup.select('a[href^="/results/event/"]')]
        df_temp = tables[0]
        df_temp['race_link'] = race_links
        return df_temp
    except Exception as e:
        print(f"Failed record {record_id}: {e}")
        return None
    
record_ids = range(1000) 
tables = Parallel(n_jobs=-1, verbose=5)(delayed(fetch_record)(rid) for rid in record_ids)

tables = [t for t in tables if t is not None]
df = pd.concat(tables, ignore_index=True)

[Parallel(n_jobs=-1)]: Using backend LokyBackend with 12 concurrent workers.
[Parallel(n_jobs=-1)]: Done  48 tasks      | elapsed:   22.4s
[Parallel(n_jobs=-1)]: Done 138 tasks      | elapsed:   58.2s
[Parallel(n_jobs=-1)]: Done 264 tasks      | elapsed:  1.7min
[Parallel(n_jobs=-1)]: Done 426 tasks      | elapsed:  2.7min
[Parallel(n_jobs=-1)]: Done 624 tasks      | elapsed:  3.8min
[Parallel(n_jobs=-1)]: Done 858 tasks      | elapsed:  5.1min
[Parallel(n_jobs=-1)]: Done 1000 out of 1000 | elapsed:  5.2min finished


In [3]:
df.head()

,Rank,Name,Time,Race,Nationality,Division,Official,Beer,Container,Notes,race_link
0,1,Corey Bellemore,4:27.10,View,Canada,Men - All Ages,Official,Flying Monkeys,Bottle,🗒New World Record!,https://www.beermile.com/results/event/5070
1,2,Chris Robertson,4:37.40,View,United States,Men - All Ages,Official,Blue Moon,Bottle,"🗒New American Record. Splits (beer,lap): (5, 1...",https://www.beermile.com/results/event/4112
2,3,Lewis Kent,4:47.17,View,Canada,Men - All Ages,Official,Amsterdam Blonde,Bottle,NaN,https://www.beermile.com/results/event/2670
3,4,Dale Clutterbuck,4:47.39,View,United States,Men - All Ages,Official,Budweiser,Bottle,"🗒England, National Record, European Record",https://www.beermile.com/results/event/2922
4,5,Brandon Shirck,4:47.72,View,United States,Men - All Ages,Official,Bud Light Platinum,Bottle,🗒3 Ounces remaining,https://www.beermile.com/results/event/2897


## 2. Extract location and date using the race link

In [4]:
def get_date(race_link):
    r = requests.get(race_link)
    soup = BeautifulSoup(r.text, "html.parser")

    date_patterns = [r'(\d{1,2}/\d{1,2}/\d{4})', r'(\d{4}-\d{2}-\d{2})', r'(\d{1,2}-\d{1,2}-\d{4})']
    page_text = soup.get_text()

    found_dates = []
    for pattern in date_patterns:
        matches = re.findall(pattern, page_text)
        found_dates.extend(matches)
    date_valid = found_dates[0] if found_dates else np.nan

    loc_pattern = r'([A-Z][a-zA-Z\s]+,\s*[A-Z][a-zA-Z\s]+)'
    loc_match = re.search(loc_pattern, page_text)
    location = loc_match.group(1).strip() if loc_match else np.nan

    return [race_link, date_valid, location]

dates_and_locations = Parallel(n_jobs=-1, verbose=5)(delayed(get_date)(race_link) for race_link in df.race_link.unique())

[Parallel(n_jobs=-1)]: Using backend LokyBackend with 12 concurrent workers.
[Parallel(n_jobs=-1)]: Done  48 tasks      | elapsed:   15.0s
[Parallel(n_jobs=-1)]: Done 138 tasks      | elapsed:   31.8s
[Parallel(n_jobs=-1)]: Done 264 tasks      | elapsed:  1.0min
[Parallel(n_jobs=-1)]: Done 426 tasks      | elapsed:  1.7min
[Parallel(n_jobs=-1)]: Done 624 tasks      | elapsed:  2.7min
[Parallel(n_jobs=-1)]: Done 858 tasks      | elapsed:  4.0min
[Parallel(n_jobs=-1)]: Done 1128 tasks      | elapsed:  5.3min
[Parallel(n_jobs=-1)]: Done 1223 out of 1223 | elapsed:  5.8min finished


In [5]:
# merge with the original dataset
dates_and_locations_df = pd.DataFrame(dates_and_locations, columns=['race_link','date','location'])
dates_and_locations_df['date'] = pd.to_datetime(dates_and_locations_df['date'])

df = pd.merge(df, dates_and_locations_df, on='race_link', how='left')
df.head()

,Rank,Name,Time,Race,Nationality,Division,Official,Beer,Container,Notes,race_link,date,location
0,1,Corey Bellemore,4:27.10,View,Canada,Men - All Ages,Official,Flying Monkeys,Bottle,🗒New World Record!,https://www.beermile.com/results/event/5070,2025-07-26,"Lisbon, Portugal"
1,2,Chris Robertson,4:37.40,View,United States,Men - All Ages,Official,Blue Moon,Bottle,"🗒New American Record. Splits (beer,lap): (5, 1...",https://www.beermile.com/results/event/4112,2020-09-27,NaN
2,3,Lewis Kent,4:47.17,View,Canada,Men - All Ages,Official,Amsterdam Blonde,Bottle,NaN,https://www.beermile.com/results/event/2670,2015-12-02,"Austin, TX"
3,4,Dale Clutterbuck,4:47.39,View,United States,Men - All Ages,Official,Budweiser,Bottle,"🗒England, National Record, European Record",https://www.beermile.com/results/event/2922,2016-08-01,"London, United KingdomBeer Mile World Classic Men"
4,5,Brandon Shirck,4:47.72,View,United States,Men - All Ages,Official,Bud Light Platinum,Bottle,🗒3 Ounces remaining,https://www.beermile.com/results/event/2897,2016-07-17,"Ojai, CA"


### 2.1 Clean location variable

In [6]:
def clean_location(loc):
    if pd.isna(loc):
        return np.nan

    loc = loc.strip()

    # Case 1: U.S. city pattern — stop after state code (two uppercase letters after comma)
    m_us = re.match(r'^(.+?,\s*[A-Z]{2})(?:[^A-Za-z]|$)', loc)
    if not m_us:
        # If the above fails (because text continues directly, like CAWVTC), fix it manually
        m_us = re.match(r'^(.+?,\s*[A-Z]{2})(?=[A-Z])', loc)
    if m_us:
        return m_us.group(1).strip()

    # Case 2: General international pattern — split when a lowercase letter is immediately followed by a comma and an uppercase word, 
    # then another uppercase after lowercase (like "United KingdomWolrdRacing")
    m_int = re.match(r'^(.+?,\s*[A-Z][a-z]+(?:\s+[A-Z][a-z]+)*)', loc)
    if m_int:
        # remove junk after country name
        cleaned = re.split(r'(?<=[a-z])(?=[A-Z])', m_int.group(1))[0]
        return cleaned.strip()

    # Fallback: try splitting on pattern lowercase→Uppercase after a comma
    cleaned = re.split(r'(?<=[a-z])(?=[A-Z])', loc)[0]
    return cleaned.strip(' ,.-')

df['clean_location'] = df['location'].apply(clean_location)

In [7]:
df[['location','clean_location']].head()

,location,clean_location
0,"Lisbon, Portugal","Lisbon, Portugal"
1,NaN,NaN
2,"Austin, TX","Austin, TX"
3,"London, United KingdomBeer Mile World Classic Men","London, United Kingdom"
4,"Ojai, CA","Ojai, CA"


### 2.2 Extract coordinates of each race event

In [8]:
unique_locations = df[['clean_location']].drop_duplicates(subset='clean_location').dropna()

def get_coordinates(i, location):
    geolocator = ArcGIS(user_agent=f"temp_app_{i}_v1", timeout=10) 
    
    try:
        loc = geolocator.geocode(location)
        if loc:
            return [location, loc.latitude, loc.longitude]
        else:
            return [location, None, None]
    except Exception as e:
        print(f"Error geocoding {location}: {e}")
        return [location, None, None]

coords = [get_coordinates(i, loc) for i, loc in enumerate(unique_locations.clean_location)]

In [9]:
coords_df = pd.DataFrame(coords, columns=['clean_location','lat','lon'])
df = pd.merge(df, coords_df, on='clean_location')

print(f'Percentage of missing locations ({df.shape[0]} total records)')
df[['lon','lat']].isna().sum() / df.shape[0]

Percentage of missing locations (8470 total records)


lon    0.002715
lat    0.002715
dtype: float64

In [10]:
df.head()

,Rank,Name,Time,Race,Nationality,Division,Official,Beer,Container,Notes,race_link,date,location,clean_location,lat,lon
0,1,Corey Bellemore,4:27.10,View,Canada,Men - All Ages,Official,Flying Monkeys,Bottle,🗒New World Record!,https://www.beermile.com/results/event/5070,2025-07-26,"Lisbon, Portugal","Lisbon, Portugal",38.707009,-9.135641
1,3,Lewis Kent,4:47.17,View,Canada,Men - All Ages,Official,Amsterdam Blonde,Bottle,NaN,https://www.beermile.com/results/event/2670,2015-12-02,"Austin, TX","Austin, TX",30.264979,-97.746598
2,4,Dale Clutterbuck,4:47.39,View,United States,Men - All Ages,Official,Budweiser,Bottle,"🗒England, National Record, European Record",https://www.beermile.com/results/event/2922,2016-08-01,"London, United KingdomBeer Mile World Classic Men","London, United Kingdom",51.507408,-0.127699
3,5,Brandon Shirck,4:47.72,View,United States,Men - All Ages,Official,Bud Light Platinum,Bottle,🗒3 Ounces remaining,https://www.beermile.com/results/event/2897,2016-07-17,"Ojai, CA","Ojai, CA",34.447966,-119.242981
4,6,Corey Gallagher,4:48.62,View,Canada,Men - All Ages,Official,Bud Light Platinum,Bottle,NaN,https://www.beermile.com/results/event/2670,2015-12-02,"Austin, TX","Austin, TX",30.264979,-97.746598


## 3. Extract weather on race day

In [ ]:
df['days_since_1970'] = (df['date'] - pd.Timestamp('1970-01-01')).dt.days

print('Unique race events:', np.unique(df[['lon','lat','days_since_1970']].to_numpy(), axis=0).shape[0])

uniques, index = np.unique(df[['lon','lat','days_since_1970']].to_numpy(), axis=0, return_index=True)
df_unique_spacetime = df.iloc[index, :].copy()

Unique race events:  1071


In [ ]:
def get_var_openmeteo(i, st, varname):
    lon, lat, date = st[0], st[1], st[2]
    base_url = "https://archive-api.open-meteo.com/v1/archive"
    
    params = {
        'latitude': lat,
        'longitude': lon,
        'start_date': date.strftime('%Y-%m-%d'),
        'end_date': date.strftime('%Y-%m-%d'),
        'daily': varname,
        'timezone': 'auto'
    }
    
    try:
        response = requests.get(base_url, params=params)
        data = response.json()
        
        if 'daily' in data and data['daily'] and varname in data['daily']:
            temp = data['daily'][varname][0]
            return temp
        else:
            return np.nan
    except:
        return np.nan

In [14]:
temperature = Parallel(n_jobs=-1, verbose=5)(delayed(get_var_openmeteo)(i, st, 'temperature_2m_mean') for i, st in enumerate(df_unique_spacetime[['lon','lat','date']].values))

[Parallel(n_jobs=-1)]: Using backend LokyBackend with 12 concurrent workers.
[Parallel(n_jobs=-1)]: Done  48 tasks      | elapsed:   10.7s
[Parallel(n_jobs=-1)]: Done 138 tasks      | elapsed:   27.7s
[Parallel(n_jobs=-1)]: Done 264 tasks      | elapsed:   54.7s
[Parallel(n_jobs=-1)]: Done 426 tasks      | elapsed:  1.7min
[Parallel(n_jobs=-1)]: Done 624 tasks      | elapsed:  2.5min
[Parallel(n_jobs=-1)]: Done 858 tasks      | elapsed:  3.7min
[Parallel(n_jobs=-1)]: Done 1071 out of 1071 | elapsed:  4.8min finished


In [15]:
humidity = Parallel(n_jobs=-1, verbose=5)(delayed(get_var_openmeteo)(i, st, 'relative_humidity_2m_mean') for i, st in enumerate(df_unique_spacetime[['lon','lat','date']].values))

[Parallel(n_jobs=-1)]: Using backend LokyBackend with 12 concurrent workers.
[Parallel(n_jobs=-1)]: Done  48 tasks      | elapsed:   14.8s
[Parallel(n_jobs=-1)]: Done 138 tasks      | elapsed:   45.2s
[Parallel(n_jobs=-1)]: Done 264 tasks      | elapsed:  1.4min
[Parallel(n_jobs=-1)]: Done 426 tasks      | elapsed:  2.1min
[Parallel(n_jobs=-1)]: Done 624 tasks      | elapsed:  3.1min
[Parallel(n_jobs=-1)]: Done 858 tasks      | elapsed:  4.3min
[Parallel(n_jobs=-1)]: Done 1071 out of 1071 | elapsed:  5.4min finished


In [16]:
precip = Parallel(n_jobs=-1, verbose=5)(delayed(get_var_openmeteo)(i, st, 'precipitation_sum') for i, st in enumerate(df_unique_spacetime[['lon','lat','date']].values))

[Parallel(n_jobs=-1)]: Using backend LokyBackend with 12 concurrent workers.
[Parallel(n_jobs=-1)]: Done  48 tasks      | elapsed:   14.8s
[Parallel(n_jobs=-1)]: Done 138 tasks      | elapsed:   44.1s
[Parallel(n_jobs=-1)]: Done 264 tasks      | elapsed:  1.4min
[Parallel(n_jobs=-1)]: Done 426 tasks      | elapsed:  2.2min
[Parallel(n_jobs=-1)]: Done 624 tasks      | elapsed:  3.2min
[Parallel(n_jobs=-1)]: Done 858 tasks      | elapsed:  4.3min
[Parallel(n_jobs=-1)]: Done 1071 out of 1071 | elapsed:  5.4min finished


In [17]:
df_unique_spacetime['temperature'] = temperature
df_unique_spacetime['humidity'] = humidity
df_unique_spacetime['precipitation'] = precip

In [18]:
# merge weahter data to original dataset
df = pd.merge(df, df_unique_spacetime[['lon','lat','days_since_1970','temperature','humidity','precipitation']], on=['lon','lat','days_since_1970'])

## 4. Drop unnecessary columns

In [19]:
df.head()

,Rank,Name,Time,Race,Nationality,Division,Official,Beer,Container,Notes,race_link,date,location,clean_location,lat,lon,days_since_1970,temperature,humidity,precipitation
0,1,Corey Bellemore,4:27.10,View,Canada,Men - All Ages,Official,Flying Monkeys,Bottle,🗒New World Record!,https://www.beermile.com/results/event/5070,2025-07-26,"Lisbon, Portugal","Lisbon, Portugal",38.707009,-9.135641,20295,28.1,35.0,0.0
1,3,Lewis Kent,4:47.17,View,Canada,Men - All Ages,Official,Amsterdam Blonde,Bottle,NaN,https://www.beermile.com/results/event/2670,2015-12-02,"Austin, TX","Austin, TX",30.264979,-97.746598,16771,10.7,74.0,0.0
2,4,Dale Clutterbuck,4:47.39,View,United States,Men - All Ages,Official,Budweiser,Bottle,"🗒England, National Record, European Record",https://www.beermile.com/results/event/2922,2016-08-01,"London, United KingdomBeer Mile World Classic Men","London, United Kingdom",51.507408,-0.127699,17014,15.2,79.0,12.7
3,5,Brandon Shirck,4:47.72,View,United States,Men - All Ages,Official,Bud Light Platinum,Bottle,🗒3 Ounces remaining,https://www.beermile.com/results/event/2897,2016-07-17,"Ojai, CA","Ojai, CA",34.447966,-119.242981,16999,21.4,69.0,0.0
4,6,Corey Gallagher,4:48.62,View,Canada,Men - All Ages,Official,Bud Light Platinum,Bottle,NaN,https://www.beermile.com/results/event/2670,2015-12-02,"Austin, TX","Austin, TX",30.264979,-97.746598,16771,10.7,74.0,0.0


In [21]:
df_final = df.drop(columns=['Rank','Name','Race','Notes','race_link','days_since_1970','location','clean_location']).copy()
df_final.head()

,Time,Nationality,Division,Official,Beer,Container,date,lat,lon,temperature,humidity,precipitation
0,4:27.10,Canada,Men - All Ages,Official,Flying Monkeys,Bottle,2025-07-26,38.707009,-9.135641,28.1,35.0,0.0
1,4:47.17,Canada,Men - All Ages,Official,Amsterdam Blonde,Bottle,2015-12-02,30.264979,-97.746598,10.7,74.0,0.0
2,4:47.39,United States,Men - All Ages,Official,Budweiser,Bottle,2016-08-01,51.507408,-0.127699,15.2,79.0,12.7
3,4:47.72,United States,Men - All Ages,Official,Bud Light Platinum,Bottle,2016-07-17,34.447966,-119.242981,21.4,69.0,0.0
4,4:48.62,Canada,Men - All Ages,Official,Bud Light Platinum,Bottle,2015-12-02,30.264979,-97.746598,10.7,74.0,0.0


In [22]:
# save
df_final.to_csv('../data/beer_mile_data.csv')